# Role C — RFM Customer Segmentation

This notebook documents the Role C part of the Week 7 group project: customer-level RFM segmentation with Python and pandas.

RFM means:

- **Recency:** days since the customer's last purchase.
- **Frequency:** number of purchases.
- **Monetary:** total customer spend.

Eesmärk: muuta tellimuste tasemel müügiandmed kliendi tasemel RFM tabeliks, anda skoorid ja määrata kliendisegmendid.

## 1. Setup and Safe Supabase Connection

This setup follows the same safe Week 7 pattern: credentials are loaded from the local `.env` file, but they are never printed in the notebook.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
from supabase import create_client

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

env_path = Path('../../.env')
load_dotenv(env_path)

SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY') or os.getenv('SUPABASE_ANON_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError('Missing Supabase settings in week7-python/.env')

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print('Supabase client ready. Credentials were loaded but not printed.')

## 2. Load or Reuse Cleaned Sales Data

Role C needs cleaned sales transactions. If `df_sales_clean` already exists in the notebook session, this notebook reuses it. Otherwise, it loads the `sales` table from Supabase and prepares the columns needed for RFM.

Only non-sensitive sales columns are displayed.

In [ ]:
def fetch_table(table_name, page_size=1000):
    rows = []
    start = 0

    while True:
        end = start + page_size - 1
        response = supabase.table(table_name).select('*').range(start, end).execute()
        batch = response.data or []
        rows.extend(batch)

        if len(batch) < page_size:
            break

        start += page_size

    return pd.DataFrame(rows)


if 'df_sales_clean' in globals():
    sales = df_sales_clean.copy()
    print('Using existing df_sales_clean from this notebook session.')
else:
    sales = fetch_table('sales')
    print('Loaded sales table from Supabase.')

print('Sales shape:', sales.shape)

public_columns = [
    'sale_id', 'customer_id', 'sale_date', 'product_id', 'quantity',
    'unit_price', 'total_price', 'store_location', 'channel'
]
display(sales[[column for column in public_columns if column in sales.columns]].head())

## 3. Validate Required Columns

The RFM workflow requires:

- `customer_id` for customer-level grouping.
- `sale_id` for purchase counting.
- `sale_date` for recency.
- `total_price` for monetary value.

If any required column is missing, the notebook stops with a clear error.

In [ ]:
required_columns = ['customer_id', 'sale_id', 'sale_date', 'total_price']
missing_columns = [column for column in required_columns if column not in sales.columns]

if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

print('Required columns are present:', required_columns)

## 4. Prepare Sales Data for RFM

This section keeps the RFM input focused. It converts dates and prices to usable types, removes rows missing required values, and keeps only positive sales values.

Eesti keeles: RFM analüüs vajab usaldusväärseid tehinguid, kus klient, kuupäev ja müügisumma on olemas.

In [ ]:
sales_clean = sales[required_columns].copy()

sales_clean['sale_date'] = pd.to_datetime(sales_clean['sale_date'], errors='coerce')
sales_clean['total_price'] = pd.to_numeric(sales_clean['total_price'], errors='coerce')

before_rows = len(sales_clean)
sales_clean = sales_clean.dropna(subset=required_columns).copy()
sales_clean = sales_clean[sales_clean['total_price'] > 0].copy()
after_rows = len(sales_clean)

print('Rows before cleaning:', before_rows)
print('Rows after cleaning:', after_rows)
print('Unique customers:', sales_clean['customer_id'].nunique())
print('Date range:', sales_clean['sale_date'].min().date(), 'to', sales_clean['sale_date'].max().date())

display(sales_clean.head())

## 5. Calculate RFM Metrics

This section converts order-level sales data into customer-level RFM metrics. Each customer gets one row with Recency, Frequency, and Monetary values.

The workbook uses a fixed reference date, but this notebook uses the day after the latest sale in the dataset so the analysis stays consistent if the dataset changes.

### Recency

Recency = days since the customer's last purchase. Lower recency means the customer bought more recently.

In [ ]:
analysis_date = sales_clean['sale_date'].max() + pd.Timedelta(days=1)

recency = (
    sales_clean.groupby('customer_id')
    .agg(last_purchase_date=('sale_date', 'max'))
    .reset_index()
)
recency['recency_days'] = (analysis_date - recency['last_purchase_date']).dt.days

print('Analysis date:', analysis_date.date())
display(recency.head())

### Frequency

Frequency = number of purchases. Higher frequency means the customer bought more often.

In [ ]:
frequency = (
    sales_clean.groupby('customer_id')
    .agg(frequency=('sale_id', 'count'))
    .reset_index()
)

display(frequency.head())

### Monetary

Monetary = total customer spend. Higher monetary value means the customer generated more revenue.

In [ ]:
monetary = (
    sales_clean.groupby('customer_id')
    .agg(monetary_value=('total_price', 'sum'))
    .reset_index()
)

display(monetary.head())

## 6. Merge RFM Metrics

This step merges Recency, Frequency, and Monetary into one customer-level table.

In [ ]:
rfm = (
    recency.merge(frequency, on='customer_id', how='inner')
    .merge(monetary, on='customer_id', how='inner')
)

print('RFM shape:', rfm.shape)
display(rfm.head())

## 7. Assign R/F/M Scores

`pd.qcut` divides customers into ranked groups. This notebook ranks values first and uses `duplicates='drop'` so the scoring step is safer if duplicate values appear.

- Recency: fewer days is better.
- Frequency: more purchases is better.
- Monetary: higher spend is better.

In [ ]:
def qcut_score(series, higher_is_better=True, q=5):
    ranked = series.rank(method='first')
    bin_numbers = pd.qcut(ranked, q=q, labels=False, duplicates='drop')
    max_bin = int(bin_numbers.max()) + 1

    if higher_is_better:
        return (bin_numbers + 1).astype(int)

    return (max_bin - bin_numbers).astype(int)


rfm['R_score'] = qcut_score(rfm['recency_days'], higher_is_better=False)
rfm['F_score'] = qcut_score(rfm['frequency'], higher_is_better=True)
rfm['M_score'] = qcut_score(rfm['monetary_value'], higher_is_better=True)

score_columns = ['R_score', 'F_score', 'M_score']
missing_score_columns = [column for column in score_columns if column not in rfm.columns]

if missing_score_columns:
    raise ValueError(f'Missing score columns: {missing_score_columns}')

if rfm[score_columns].isnull().any().any():
    raise ValueError('R/F/M scores contain missing values')

display(rfm[['customer_id', 'recency_days', 'frequency', 'monetary_value', 'R_score', 'F_score', 'M_score']].head())

## 8. Create RFM_Score

`RFM_Score` is the sum of the three scores. The minimum expected score is 3 and the maximum expected score is 15.

In [ ]:
rfm['RFM_Score'] = rfm[['R_score', 'F_score', 'M_score']].sum(axis=1)

print('RFM score range:', rfm['RFM_Score'].min(), 'to', rfm['RFM_Score'].max())
display(rfm.sort_values('RFM_Score', ascending=False).head(10))

## 9. Assign Customer Segments

The workbook segment labels are based on the total `RFM_Score`:

- 13–15 = VIP Champions
- 10–12 = Loyal Customers
- 7–9 = Potential Loyalists
- 4–6 = At Risk
- 3 = Lost

In [ ]:
def assign_segment(score):
    if score >= 13:
        return 'VIP Champions'
    if score >= 10:
        return 'Loyal Customers'
    if score >= 7:
        return 'Potential Loyalists'
    if score >= 4:
        return 'At Risk'
    return 'Lost'


rfm['Segment'] = rfm['RFM_Score'].apply(assign_segment)

if rfm['Segment'].isnull().any():
    raise ValueError('Some customers did not receive a Segment')

display(rfm[['customer_id', 'RFM_Score', 'Segment', 'recency_days', 'frequency', 'monetary_value']].head(10))

## 10. Create Segment Summary

The segment summary turns customer-level RFM results into a business-level view. It shows how many customers are in each segment and how much revenue each segment represents.

In [ ]:
segment_summary = (
    rfm.groupby('Segment')
    .agg(
        customers=('customer_id', 'count'),
        avg_recency_days=('recency_days', 'mean'),
        avg_frequency=('frequency', 'mean'),
        total_revenue=('monetary_value', 'sum'),
        avg_monetary_value=('monetary_value', 'mean'),
    )
    .reset_index()
)

if segment_summary.empty:
    raise ValueError('segment_summary is empty')

segment_summary['customer_share_pct'] = segment_summary['customers'] / segment_summary['customers'].sum() * 100
segment_summary['revenue_share_pct'] = segment_summary['total_revenue'] / segment_summary['total_revenue'].sum() * 100
segment_summary = segment_summary.sort_values('total_revenue', ascending=False)

display(segment_summary.round(2))

## Business interpretation

- **VIP Champions** are high-value customers with strong recent activity, purchase frequency, and total spend.
- **At Risk** customers still have value, but their RFM pattern suggests they may need reactivation before they become inactive.
- Marko could use these segments for targeted campaigns, such as VIP benefits, loyalty offers, or win-back messages.
- RFM is useful because it turns transaction data into practical campaign groups without needing sensitive contact details in the notebook.
- Segment summaries help compare customer count and revenue impact before choosing marketing actions.